# 1. SAS 파일 합치기 ──────────────────────────────────────

In [3]:
import numpy as np
import pandas as pd

In [4]:
files = [
    "../data/hn21_all.sas7bdat",
    "../data/hn22_all.sas7bdat",
    "../data/hn23_all.sas7bdat",
    "../data/hn24_all.sas7bdat",
]
dfs = [pd.read_sas(f, encoding="latin1") for f in files]
raw = pd.concat(dfs, ignore_index=True)
print(f"원본: {raw.shape}")

use_cols = ["age", "sex", "HE_ht", "HE_wt", "HE_BMI", "HE_wc",
            "BD1_11", "BD2_1", "BD2_31",
            "pa_aerobic",
            "sm_presnt",
            "DI1_pr", "HE_HP",
            "BD9_1", "BD9_2"]
df = raw[use_cols].copy()

원본: (27281, 1185)


# 2. 결측 코드 처리 ────────────────────────────────────────


In [3]:
for col in ["BD1_11", "BD2_1", "BD2_31", "BD9_1", "BD9_2", "DI1_pr", "HE_HP"]:
    df[col] = df[col].replace({8.0: np.nan, 9.0: np.nan})


# 3. 피처 변환 (NHANES와 동일한 카테고리 문자열) ──────────


In [9]:
df["나이"]     = df["age"]
df["키"]       = df["HE_ht"]
df["몸무게"]   = df["HE_wt"]
df["BMI"]      = df["HE_BMI"].round(1)
df["허리둘레"] = df["HE_wc"]

# 성별
df["성별"] = df["sex"].map({1.0: "남성", 2.0: "여성"})

# 음주여부: BD1_11 있으면 음주함
df["음주여부"] = df["BD1_11"].apply(lambda x: "음주함" if pd.notna(x) else "음주안함")

# 주당음주빈도
freq_map = {1.0: 7.0, 2.0: 4.5, 3.0: 2.5, 4.0: 1.0, 5.0: 0.58, 6.0: 0.23}
df["주당음주빈도"] = df["BD1_11"].map(freq_map).fillna(0.0)

# 1회음주량 (잔 중간값)
amount_map = {1.0: 1.5, 2.0: 3.5, 3.0: 5.5, 4.0: 8.0, 5.0: 10.0}
df["1회음주량"] = df["BD2_1"].map(amount_map).fillna(0.0)

# 월폭음빈도
binge_map = {1.0: 30.0, 2.0: 19.5, 3.0: 10.8, 4.0: 2.5, 5.0: 0.5}
df["월폭음빈도"] = df["BD2_31"].map(binge_map).fillna(0.0)

# 운동여부
df["운동여부"] = df["pa_aerobic"].map({1.0: "운동함", 0.0: "운동안함"})

# 주당운동횟수: 운동하면 평균 3회
df["주당운동횟수"] = df["pa_aerobic"].map({1.0: 3, 0.0: 0})

# 흡연여부 / 현재흡연여부
df["흡연여부"]     = df["sm_presnt"].map({1.0: "흡연경험있음", 0.0: "흡연경험없음"})
df["현재흡연여부"] = df["sm_presnt"].map({1.0: "매일", 0.0: "안함"})

# 당뇨: DI1_pr (1=있음, 2=없음, 확장없음)
df["당뇨진단여부"] = df["DI1_pr"].map({1.0: "있음", 2.0: "없음"}).fillna("없음")

# 고혈압: HE_HP (1=있음, 2=없음)
df["고혈압진단여부"] = df["HE_HP"].map({1.0: "있음", 2.0: "없음"}).fillna("없음")

# 수면장애: BD9_1 (1=있음, 2=없음)
df["수면장애여부"] = df["BD9_1"].map({1.0: "있음", 2.0: "없음"}).fillna("없음")

# 평균수면시간: BD9_2 카테고리 → 시간 중간값
sleep_map = {1.0: 5.0, 2.0: 6.0, 3.0: 7.0, 4.0: 8.5}
df["평균수면시간"] = df["BD9_2"].map(sleep_map).fillna(7.0)

# 식습관자가평가: 영양 컬럼 없으므로 "보통" 고정
df["식습관자가평가"] = "보통"

In [10]:
# 음주 피처 정합성 보정
non_drinker = (df["음주여부"] == "음주안함") | (df["1회음주량"] == 0)
df.loc[non_drinker, "1회음주량"]   = 0
df.loc[non_drinker, "주당음주빈도"] = 0
df.loc[non_drinker, "월폭음빈도"]  = 0

# 월폭음빈도는 실제 월음주빈도 초과 불가
df["월음주빈도"] = df["주당음주빈도"] * 4.3
df["월폭음빈도"] = df[["월폭음빈도", "월음주빈도"]].min(axis=1)
df = df.drop(columns=["월음주빈도"])

numeric_cols = ["나이", "키", "몸무게", "허리둘레",
                "1회음주량", "주당음주빈도", "월폭음빈도", "주당운동횟수", "평균수면시간"]
df[numeric_cols] = df[numeric_cols].round(2)

# 4. 최종 컬럼 선택 및 결측 제거 ─────────────────────────


In [11]:
FEATURE_NAMES = [
    "나이", "키", "몸무게", "BMI", "허리둘레",
    "1회음주량", "주당음주빈도", "월폭음빈도", "주당운동횟수", "평균수면시간",
    "성별", "음주여부", "운동여부", "흡연여부", "현재흡연여부",
    "당뇨진단여부", "고혈압진단여부", "수면장애여부",
    "식습관자가평가"
]
knhanes_clean = df[FEATURE_NAMES].dropna()
print(f"정제 후: {knhanes_clean.shape}")

knhanes_clean.to_csv("../data/knhanes_preprocessed.csv", index=False, encoding="utf-8-sig")
print("저장 완료: data/knhanes_preprocessed.csv")

정제 후: (20243, 19)
저장 완료: data/knhanes_preprocessed.csv


# 5. Pseudo-labeling — 현재 모델로 KNHANES 예측

In [8]:
import joblib
import numpy as np

model = joblib.load("../ai_worker/models/fatty_liver_model.pkl")
knhanes = pd.read_csv("../data/knhanes_preprocessed.csv")

# 1. 원본 예측 확률
proba = model.predict_proba(knhanes)  # (n, 4) — S0, S1, S2, S3 순

# 2. 사전확률 보정 (베이즈)
# NHANES 실측 분포 (S0, S1, S2, S3)
nhanes_prior = np.array([0.408, 0.120, 0.163, 0.308])
# 한국인 유병률 기반 (가이드라인: 유병률 약 30%)
korean_prior = np.array([0.70, 0.20, 0.07, 0.03])

calibrated = proba * (korean_prior / nhanes_prior)
calibrated = calibrated / calibrated.sum(axis=1, keepdims=True)

pred_class = calibrated.argmax(axis=1)
max_proba  = calibrated.max(axis=1)

print(f"전체: {len(knhanes)}건")
print(f"확률 0.8 이상: {(max_proba >= 0.8).sum()}건")
print(f"확률 0.7 이상: {(max_proba >= 0.7).sum()}건")

print("\n클래스별 분포 (보정 후 전체):")
temp = pd.DataFrame({"pred": pred_class})
print(temp["pred"].map({0:"정상",1:"경미",2:"중등도",3:"중증"}).value_counts())

c:\Users\sunje\Desktop\OZ\AI_Final_Projects\AI_02_03\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


전체: 20243건
확률 0.8 이상: 9513건
확률 0.7 이상: 11781건

클래스별 분포 (보정 후 전체):
pred
정상     17953
경미      1999
중등도      174
중증       117
Name: count, dtype: int64


In [9]:
CONFIDENCE_THRESHOLD = 0.8

mask = max_proba >= CONFIDENCE_THRESHOLD
knhanes_pseudo = knhanes[mask].copy()
knhanes_pseudo["지방간단계"] = pred_class[mask]
knhanes_pseudo["지방간단계명"] = knhanes_pseudo["지방간단계"].map({
    0: "정상", 1: "경미", 2: "중등도", 3: "중증"
})

print(f"채택: {len(knhanes_pseudo)}건 / 전체 {len(knhanes)}건 ({len(knhanes_pseudo)/len(knhanes)*100:.1f}%)")
print("\n클래스별 분포 (채택):")
print(knhanes_pseudo["지방간단계명"].value_counts())

채택: 9513건 / 전체 20243건 (47.0%)

클래스별 분포 (채택):
지방간단계명
정상    9491
경미      22
Name: count, dtype: int64


In [6]:
nhanes = pd.read_csv("../data/nhanes_preprocessed.csv")
combined = pd.concat([nhanes, knhanes_pseudo], ignore_index=True)
combined.to_csv("../data/combined_train.csv", index=False, encoding="utf-8-sig")

print(f"NHANES:          {len(nhanes):,}건")
print(f"KNHANES pseudo:  {len(knhanes_pseudo):,}건")
print(f"합계:            {len(combined):,}건")
print("\n지방간단계 분포:")
print(combined["지방간단계명"].value_counts())

NHANES:          15,646건
KNHANES pseudo:  7,325건
합계:            22,971건

지방간단계 분포:
지방간단계명
정상     13535
중증      4997
중등도     2553
경미      1886
Name: count, dtype: int64


In [5]:
nhanes = pd.read_csv("../data/nhanes_preprocessed.csv")
print(nhanes["지방간단계명"].value_counts(normalize=True).round(3) * 100)


지방간단계명
정상     40.8
중증     30.8
중등도    16.3
경미     12.0
Name: proportion, dtype: float64
